# 11 — P2: Lipschitz Bound Computation for ViT-Tiny

**Plan 2 — Phase 2.** Compositional upper bound on the global Lipschitz constant of the
trained ViT-Tiny (input pixels → logit vector). Used by Phase 6 as the cheapest
verification stage.

## Per-component bounds
| Component | Lipschitz upper bound |
|---|---|
| `Linear(W,b)` | `σ_max(W)` (largest singular value) |
| `Conv2d` (stride=kernel, non-overlapping patches) | `σ_max(W_reshape)` after reshape to `(out, in·kH·kW)` |
| `RMSNorm(γ, ε)` | `‖γ‖_∞ · √(D/ε)` |
| `ReLU` | `1` |
| Mean-pool over N tokens | `1/√N` |

## MHSA per-head (Kim et al. 2021)
```
L_head ≤ N^(3/2) · σ(W_Q) · σ(W_K) · σ(W_V) / √head_dim   +   √N · σ(W_V)
L_MHSA ≤ σ(W_O) · √( Σ_h L_head[h]² )
```

## Block (Pre-LN with residual)
```
L_block ≤ ( 1 + L_RMSNorm · L_MHSA ) · ( 1 + L_RMSNorm · L_MLP )
```
(the `1+` comes from the identity branch of the residual)

## Full network
```
L_total ≤ L_patch_embed · ∏ L_block · L_RMSNorm_final · (1/√N) · L_head
```

## L∞ → L2 conversion (used in Phase 6 pre-filter)
For an L∞ perturbation of radius `ε` on a `D`-dim input, the worst-case L2 displacement is `ε·√D`.
A sample with clean logit margin `m = z_y − max_{c≠y} z_c` is **Lipschitz-certified** if:
```
m  >  √2 · L_total · ε · √D
```
(The `√2` factor comes from the worst-case Lipschitz of the difference `z_y − z_c`,
which is a 2-component linear combination of the logit vector; we pick the conservative
constant rather than the plan's tighter `m > L · ε · √D` to keep soundness margin.)

In [1]:
!pip install -q numpy torch torchvision tqdm pyyaml

In [2]:
# ── Imports ───────────────────────────────────────────────────────────────
from __future__ import annotations
import math, json, warnings
from pathlib import Path
from typing import Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

Device : cuda


In [3]:
# ── ViT-Tiny class (must match notebooks 09 / 10 byte-for-byte) ───────────
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.dim, self.eps = dim, eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)

class PatchEmbed(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, embed_dim=64):
        super().__init__()
        self.img_size, self.patch_size = img_size, patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class MHSA(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.scale    = self.head_dim ** -0.5
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=True)
    def forward(self, x):
        B, N, C = x.shape
        H, D = self.num_heads, self.head_dim
        q = self.W_q(x).view(B, N, H, D).transpose(1, 2)
        k = self.W_k(x).view(B, N, H, D).transpose(1, 2)
        v = self.W_v(x).view(B, N, H, D).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn   = F.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.W_o(out)

class MLPBlock(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=2):
        super().__init__()
        h = embed_dim * mlp_ratio
        self.fc1, self.fc2 = nn.Linear(embed_dim, h), nn.Linear(h, embed_dim)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim, eps_rms)
        self.attn  = MHSA(embed_dim, num_heads)
        self.norm2 = RMSNorm(embed_dim, eps_rms)
        self.mlp   = MLPBlock(embed_dim, mlp_ratio)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class ViTTiny(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, num_classes=10,
                 embed_dim=64, num_heads=2, num_layers=2, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.cfg = dict(img_size=img_size, patch_size=patch_size,
                        in_channels=in_channels, num_classes=num_classes,
                        embed_dim=embed_dim, num_heads=num_heads,
                        num_layers=num_layers, mlp_ratio=mlp_ratio, eps_rms=eps_rms)
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.patch_embed.n_patches, embed_dim))
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, eps_rms)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(embed_dim, eps_rms)
        self.head = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.head(self.norm(x).mean(dim=1))

print('Model class defined')

Model class defined


In [4]:
# ── Per-component Lipschitz primitives ────────────────────────────────────
@torch.no_grad()
def lipschitz_linear(W: torch.Tensor) -> float:
    """σ_max(W) — operator 2-norm."""
    return float(torch.linalg.svdvals(W.detach()).max())

@torch.no_grad()
def lipschitz_conv_nonoverlap(conv: nn.Conv2d) -> float:
    """
    Lipschitz of a stride==kernel conv (non-overlapping patches).
    Equivalent to a linear map W_reshape with shape (out_C, in_C·kH·kW)
    applied independently to each patch — the operator norm equals
    σ_max of that reshape.
    """
    assert conv.stride == (conv.kernel_size[0], conv.kernel_size[1]), \
        'Lipschitz primitive expects stride == kernel (non-overlapping)'
    W = conv.weight.detach()                                 # (out_C, in_C, kH, kW)
    return float(torch.linalg.svdvals(W.reshape(W.shape[0], -1)).max())

def lipschitz_rmsnorm(gamma: torch.Tensor, eps_rms: float, dim: int) -> float:
    """
    Worst-case Lipschitz of  y_i = γ_i · x_i / sqrt( mean(x²) + ε ).
    Bound: ‖γ‖_∞ · √(D / ε).  Loose in practice but sound.
    """
    return float(gamma.detach().abs().max() * math.sqrt(dim / eps_rms))

def lipschitz_relu() -> float:
    return 1.0

def lipschitz_meanpool(N: int) -> float:
    """For map (R^{N×D} → R^D) given by (1/N)·Σ_i x_i:  L ≤ 1/√N."""
    return 1.0 / math.sqrt(N)

In [5]:
# ── MHSA Lipschitz (Kim et al. 2021) ──────────────────────────────────────
@torch.no_grad()
def lipschitz_mhsa(attn: MHSA, n_tokens: int) -> Dict[str, float]:
    """
    Per-head bound (Kim et al. 2021, Lemma 3.4):
        L_head ≤ N^(3/2) · σ(W_Q) · σ(W_K) · σ(W_V) / √d  +  √N · σ(W_V)

    where d = head_dim (the 1/√d in attention scaling shows up here).
    Note: σ(W_*) here is the operator norm of the per-head slice of the
    full Q/K/V matrices. We bound conservatively by the full matrix's
    σ_max — a slice can only have a smaller singular value.

    Multi-head:  L_MHSA ≤ σ(W_O) · √( Σ_h L_head[h]² ).
    With the conservative slice bound, all heads share the same L_head.
    """
    H, D = attn.num_heads, attn.head_dim
    sQ = lipschitz_linear(attn.W_q.weight)
    sK = lipschitz_linear(attn.W_k.weight)
    sV = lipschitz_linear(attn.W_v.weight)
    sO = lipschitz_linear(attn.W_o.weight)

    L_head = (n_tokens ** 1.5) * sQ * sK * sV / math.sqrt(D) + math.sqrt(n_tokens) * sV
    L_mhsa = sO * math.sqrt(H * L_head ** 2)
    return dict(sigma_Q=sQ, sigma_K=sK, sigma_V=sV, sigma_O=sO,
                L_head=float(L_head), L_mhsa=float(L_mhsa))

In [6]:
# ── MLP-block Lipschitz ───────────────────────────────────────────────────
@torch.no_grad()
def lipschitz_mlp(mlp: MLPBlock) -> float:
    """L_MLP = σ(W2) · L_ReLU · σ(W1) = σ(W2) · σ(W1)."""
    return lipschitz_linear(mlp.fc1.weight) * lipschitz_relu() * lipschitz_linear(mlp.fc2.weight)

In [7]:
# ── Block + global Lipschitz ──────────────────────────────────────────────
@torch.no_grad()
def lipschitz_block(block: TransformerBlock, n_tokens: int, eps_rms: float) -> Dict[str, float]:
    """L_block = (1 + L_RMSNorm1 · L_MHSA) · (1 + L_RMSNorm2 · L_MLP)."""
    D = block.norm1.dim
    L_rms1 = lipschitz_rmsnorm(block.norm1.weight, eps_rms, D)
    L_rms2 = lipschitz_rmsnorm(block.norm2.weight, eps_rms, D)
    mhsa   = lipschitz_mhsa(block.attn, n_tokens)
    L_mlp  = lipschitz_mlp(block.mlp)
    L_attn_branch = 1.0 + L_rms1 * mhsa['L_mhsa']
    L_mlp_branch  = 1.0 + L_rms2 * L_mlp
    L_block       = L_attn_branch * L_mlp_branch
    return dict(L_rms1=L_rms1, L_rms2=L_rms2,
                **{f'mhsa_{k}': v for k, v in mhsa.items()},
                L_mlp=L_mlp,
                L_attn_branch=L_attn_branch, L_mlp_branch=L_mlp_branch,
                L_block=L_block)


@torch.no_grad()
def lipschitz_full(model: ViTTiny) -> Dict:
    cfg     = model.cfg
    N       = model.patch_embed.n_patches
    eps_rms = cfg['eps_rms']

    L_patch = lipschitz_conv_nonoverlap(model.patch_embed.proj)
    L_blocks = [lipschitz_block(b, N, eps_rms) for b in model.blocks]
    L_norm_final = lipschitz_rmsnorm(model.norm.weight, eps_rms, model.norm.dim)
    L_pool = lipschitz_meanpool(N)
    L_head = lipschitz_linear(model.head.weight)

    L_total = L_patch
    for b in L_blocks: L_total *= b['L_block']
    L_total *= L_norm_final * L_pool * L_head

    return dict(
        n_tokens=N, embed_dim=cfg['embed_dim'], num_heads=cfg['num_heads'],
        L_patch_embed=L_patch,
        L_blocks=L_blocks,
        L_norm_final=L_norm_final,
        L_meanpool=L_pool,
        L_head=L_head,
        L_total=float(L_total),
    )

In [8]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')


# Define source paths on Drive (adjust if paths are incorrect)
drive_base_standard = '/content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_standard'
drive_base_lipmargin = '/content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_lipmargin'  # assuming 'vit_tiny' based on CKPTS

local_base = 'runs/'

os.makedirs(local_base, exist_ok=True)

# Copy the directories
shutil.copytree(drive_base_standard, local_base + 'vit_tiny_standard')
shutil.copytree(drive_base_lipmargin, local_base + 'vit_tiny_lipmargin')

Mounted at /content/drive


'runs/vit_tiny_lipmargin'

In [9]:
# ── Loader for both checkpoints ───────────────────────────────────────────
def load_vit(ckpt_path: Path) -> ViTTiny:
    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    model   = ViTTiny(**payload['cfg']).to(device)
    # Prefer materialized state_dict (no spectral_norm parametrization wrappers)
    sd = payload.get('state_dict_materialized', payload['state_dict'])
    # Strip SN parametrization keys if present (they have 'parametrizations' in the name)
    sd = {k: v for k, v in sd.items() if 'parametrizations' not in k}
    # If checkpoint was SN-wrapped, the materialized version will already have plain weights
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing or unexpected:
        print(f'  load_state_dict — missing={len(missing)} unexpected={len(unexpected)}')
    model.eval()
    return model

In [10]:
# ── Compute and save Lipschitz reports for both checkpoints ───────────────
CKPTS = {
    'standard':  Path('runs/vit_tiny_standard/model.pt'),
    'lipmargin': Path('runs/vit_tiny_lipmargin/model.pt'),
}

reports = {}
for name, path in CKPTS.items():
    if not path.exists():
        print(f'[skip] {name}: checkpoint not found at {path}')
        continue
    print(f'\n══ {name} ══════════════════════════════')
    model  = load_vit(path)
    report = lipschitz_full(model)
    reports[name] = report

    print(f'  L_patch_embed  = {report["L_patch_embed"]:.4f}')
    for i, b in enumerate(report['L_blocks']):
        print(f'  block[{i}]  L_rms1={b["L_rms1"]:.2e}  '
              f'L_mhsa={b["mhsa_L_mhsa"]:.2f}  '
              f'L_mlp={b["L_mlp"]:.2f}  '
              f'→ L_block={b["L_block"]:.4e}')
    print(f'  L_norm_final   = {report["L_norm_final"]:.4e}')
    print(f'  L_meanpool     = {report["L_meanpool"]:.4f}')
    print(f'  L_head         = {report["L_head"]:.4f}')
    print(f'  L_total        = {report["L_total"]:.4e}')

    # Save
    out = path.parent / 'lipschitz.json'
    out.write_text(json.dumps(report, indent=2))
    print(f'  saved → {out}')

print('\n── Summary ─────────────────────────────────────────────────────────')
for name, r in reports.items():
    print(f'  {name:10s}  L_total = {r["L_total"]:.3e}')


══ standard ══════════════════════════════
  L_patch_embed  = 0.2455
  block[0]  L_rms1=8.39e+03  L_mhsa=50.60  L_mlp=0.37  → L_block=1.3197e+09
  block[1]  L_rms1=8.42e+03  L_mhsa=73.38  L_mlp=0.97  → L_block=5.0905e+09
  L_norm_final   = 1.1114e+04
  L_meanpool     = 0.1429
  L_head         = 1.2134
  L_total        = 3.1773e+21
  saved → runs/vit_tiny_standard/lipschitz.json

══ lipmargin ══════════════════════════════
  load_state_dict — missing=14 unexpected=0
  L_patch_embed  = 1.5771
  block[0]  L_rms1=8.59e+03  L_mhsa=147.16  L_mlp=1.34  → L_block=1.3979e+10
  block[1]  L_rms1=8.68e+03  L_mhsa=138.67  L_mlp=1.31  → L_block=1.3448e+10
  L_norm_final   = 1.1447e+04
  L_meanpool     = 0.1429
  L_head         = 0.7579
  L_total        = 3.6744e+23
  saved → runs/vit_tiny_lipmargin/lipschitz.json

── Summary ─────────────────────────────────────────────────────────
  standard    L_total = 3.177e+21
  lipmargin   L_total = 3.674e+23


In [11]:
# ── Sanity check vs Plan 2 success criterion ──────────────────────────────
#
# Plan 2 §2.5: "Standard >10³ (correct but useless), lipmargin <100 (useful)."
#
# Caveat: the RMSNorm bound ‖γ‖∞·√(D/ε) is extremely loose for small ε_rms
# (we use ε=1e-6, so √(D/ε) ≈ √(64·1e6) = 8000). This dominates the
# expression. The plan's <100 target therefore likely refers to the
# *attention-only* sub-bound or to an L_total measured with a larger ε_rms.
#
# We report all sub-bounds so the gap can be inspected; the global product
# is still valid and sound, just loose.

for name, r in reports.items():
    print(f'{name}: L_total = {r["L_total"]:.3e}')
    if name == 'standard'  and r['L_total'] > 1e3: print('  ✓ matches "standard >10³"')
    if name == 'lipmargin' and r['L_total'] < r.get('_compare', float('inf')):
        pass
if 'standard' in reports and 'lipmargin' in reports:
    ratio = reports['standard']['L_total'] / reports['lipmargin']['L_total']
    print(f'\nstandard / lipmargin Lipschitz ratio = {ratio:.2f}×')
    print('(Lipmargin training should produce a smaller global L; ratio ≫ 1 means SN+margin worked.)')

standard: L_total = 3.177e+21
  ✓ matches "standard >10³"
lipmargin: L_total = 3.674e+23

standard / lipmargin Lipschitz ratio = 0.01×
(Lipmargin training should produce a smaller global L; ratio ≫ 1 means SN+margin worked.)


In [ ]:
from pathlib import Path
import json

for name, report in reports.items():
    if name == 'standard':
        out = Path(drive_base_standard) / 'lipschitz.json'
    elif name == 'lipmargin':
        out = Path(drive_base_lipmargin) / 'lipschitz.json'
    else:
        continue

    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, indent=2))
    print(f'saved {name} report to {out}')